In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv('Nassau Candy Distributor.csv')

# View the first 5 rows to ensure it loaded correctly
df.head()


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,Division,Region,Product ID,Product Name,Sales,Units,Gross Profit,Cost
0,1,US-2021-103800-CHO-MIL-31000,03-01-2024,30-06-2026,Standard Class,103800,United States,Houston,Texas,77095,Chocolate,Interior,CHO-MIL-31000,Wonka Bar - Milk Chocolate,6.50,2,4.22,2.28
1,2,US-2021-112326-CHO-TRI-54000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,7.50,2,4.90,2.60
2,3,US-2021-112326-CHO-NUT-13000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,10.47,3,7.47,3.00
3,4,US-2021-112326-CHO-SCR-58000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-SCR-58000,Wonka Bar -Scrumdiddlyumptious,10.80,3,7.50,3.30
4,5,US-2021-141817-CHO-TRI-54000,05-01-2024,05-07-2026,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,Chocolate,Atlantic,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,11.25,3,7.35,3.90


In [2]:
# Convert text dates into actual datetime objects
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

# Calculate the initial lead time (Days)
df['Shipping Lead Time'] = (df['Ship Date'] - df['Order Date']).dt.days


ValueError: time data "13-01-2024" doesn't match format "%m-%d-%Y". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [3]:
# Convert text dates into actual datetime objects
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

# Calculate the initial lead time (Days)
df['Shipping Lead Time'] = (df['Ship Date'] - df['Order Date']).dt.days

ValueError: time data "13-01-2024" doesn't match format "%m-%d-%Y". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [4]:
# Convert text dates into actual datetime objects, telling pandas the day comes first
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True, format='mixed')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], dayfirst=True, format='mixed')

# Calculate the initial lead time (Days)
df['Shipping Lead Time'] = (df['Ship Date'] - df['Order Date']).dt.days

# View the result to make sure it worked
df[['Order Date', 'Ship Date', 'Shipping Lead Time']].head()


,Order Date,Ship Date,Shipping Lead Time
0,2024-01-03,2026-06-30,909
1,2024-01-04,2026-07-01,909
2,2024-01-04,2026-07-01,909
3,2024-01-04,2026-07-01,909
4,2024-01-05,2026-07-05,912


In [5]:
# Keep only rows where lead time is 0 or positive (removes negative errors)
df = df[df['Shipping Lead Time'] >= 0]

# Drop any rows that are missing a Ship Date
df = df.dropna(subset=['Ship Date'])

# Standardize geographic fields (makes all states uppercase so 'ny' and 'NY' match)
df['State/Province'] = df['State/Province'].str.upper()
df['Region'] = df['Region'].str.title()

# Check how many rows remain after cleaning
print(f"Total rows after cleaning: {len(df)}")


Total rows after cleaning: 10194


In [6]:
# Note: If your CSV does not have a 'Factory' column, you will need to map it 
# using the 'Products and Factories Correlation' mentioned in your documentation.
# Assuming you have a 'Factory' column:

df['Route_to_State'] = df['Factory'] + " -> " + df['State/Province']
df['Route_to_Region'] = df['Factory'] + " -> " + df['Region']

# Preview the new columns
df[['Factory', 'State/Province', 'Route_to_State', 'Shipping Lead Time']].head()


KeyError: 'Factory'

In [7]:
# 1. Define the correlation between your products and the factories
# Note: You need to replace 'Product A', 'Product B', etc., with the actual 
# product names or IDs from your dataset, and match them to the correct factory.
product_to_factory_map = {
    'Product A': "Lot's O' Nuts",
    'Product B': "Wicked Choccy's",
    'Product C': "Sugar Shack",
    'Product D': "Secret Factory",
    'Product E': "The Other Factory"
}

# 2. Create the 'Factory' column by mapping the dictionary to your Product Name column
df['Factory'] = df['Product Name'].map(product_to_factory_map)

# 3. Check if there are any missing mappings (products that didn't get assigned a factory)
missing_factories = df['Factory'].isna().sum()
print(f"Rows missing a factory assignment: {missing_factories}")

# 4. View the new column
df[['Product Name', 'Factory']].head()


Rows missing a factory assignment: 10194


,Product Name,Factory
0,Wonka Bar - Milk Chocolate,NaN
1,Wonka Bar - Triple Dazzle Caramel,NaN
2,Wonka Bar - Nutty Crunch Surprise,NaN
3,Wonka Bar -Scrumdiddlyumptious,NaN
4,Wonka Bar - Triple Dazzle Caramel,NaN


In [8]:
# Now this will work because 'Factory' exists!
df['Route_to_State'] = df['Factory'] + " -> " + df['State/Province']
df['Route_to_Region'] = df['Factory'] + " -> " + df['Region']


In [9]:
# 1. Define the correlation (Replace the 'Product X' placeholders with real product names)
product_to_factory_map = {
    'Product A': "Lot's O' Nuts",
    'Product B': "Wicked Choccy's",
    'Product C': "Sugar Shack",
    'Product D': "Secret Factory",
    'Product E': "The Other Factory"
}

# 2. Create the 'Factory' column by mapping the dictionary
df['Factory'] = df['Product Name'].map(product_to_factory_map)

# 3. Check if there are any missing mappings
missing_factories = df['Factory'].isna().sum()
print(f"Rows missing a factory assignment: {missing_factories}")

# Preview the mapping
df[['Product Name', 'Factory']].head()

Rows missing a factory assignment: 10194


,Product Name,Factory
0,Wonka Bar - Milk Chocolate,NaN
1,Wonka Bar - Triple Dazzle Caramel,NaN
2,Wonka Bar - Nutty Crunch Surprise,NaN
3,Wonka Bar -Scrumdiddlyumptious,NaN
4,Wonka Bar - Triple Dazzle Caramel,NaN


In [10]:
# Now that 'Factory' exists, we can create the route columns
df['Route_to_State'] = df['Factory'] + " -> " + df['State/Province']
df['Route_to_Region'] = df['Factory'] + " -> " + df['Region']

# Preview the new route columns
df[['Factory', 'State/Province', 'Route_to_State', 'Shipping Lead Time']].head()

,Factory,State/Province,Route_to_State,Shipping Lead Time
0,NaN,TEXAS,NaN,909
1,NaN,ILLINOIS,NaN,909
2,NaN,ILLINOIS,NaN,909
3,NaN,ILLINOIS,NaN,909
4,NaN,PENNSYLVANIA,NaN,912


In [11]:
# Now that 'Factory' exists, we can create the route columns
df['Route_to_State'] = df['Factory'] + " -> " + df['State/Province']
df['Route_to_Region'] = df['Factory'] + " -> " + df['Region']

# Preview the new route columns
df[['Factory', 'State/Province', 'Route_to_State', 'Shipping Lead Time']].head()

,Factory,State/Province,Route_to_State,Shipping Lead Time
0,NaN,TEXAS,NaN,909
1,NaN,ILLINOIS,NaN,909
2,NaN,ILLINOIS,NaN,909
3,NaN,ILLINOIS,NaN,909
4,NaN,PENNSYLVANIA,NaN,912


In [12]:
# Calculate Route KPIs (Aggregating by Route to State)
route_kpis = df.groupby('Route_to_State').agg(
    Average_Lead_Time=('Shipping Lead Time', 'mean'),
    Route_Volume=('Order ID', 'count')
).reset_index()

# Sort to see the 10 fastest routes (Efficiency Benchmarking)
fastest_routes = route_kpis.sort_values(by='Average_Lead_Time').reset_index(drop=True)

# Display the top 10
fastest_routes.head(10)

,Route_to_State,Average_Lead_Time,Route_Volume


In [13]:
# See a list of all the exact product names in your dataset
print(df['Product Name'].unique())


<ArrowStringArray>
[       'Wonka Bar - Milk Chocolate', 'Wonka Bar - Triple Dazzle Caramel',
 'Wonka Bar - Nutty Crunch Surprise',    'Wonka Bar -Scrumdiddlyumptious',
         'Wonka Bar - Fudge Mallows',                         'Wonka Gum',
                         'Kazookles',                'Lickable Wallpaper',
              'Fizzy Lifting Drinks',                       'Laffy Taffy',
                         'SweeTARTS',                             'Nerds',
                       'Hair Toffee',            'Everlasting Gobstopper',
                           'Fun Dip']
Length: 15, dtype: str


In [14]:
# Replace the factory names on the right with the correct ones from your instructions
product_to_factory_map = {
    'Wonka Bar - Triple Dazzle Caramel': "Lot's O' Nuts",   # <-- Example
    'Wonka Bar - Nutty Crunch Surprise': "Sugar Shack",     # <-- Example
    'Wonka Bar -Scrumdiddlyumptious': "Secret Factory",     # <-- Example
    # ... add the rest of your unique products here
}

# Apply the mapping again
df['Factory'] = df['Product Name'].map(product_to_factory_map)

# Re-create the routes now that Factory has real text in it
df['Route_to_State'] = df['Factory'] + " -> " + df['State/Province']

# Check the results
df[['Product Name', 'Factory', 'Route_to_State']].head()

,Product Name,Factory,Route_to_State
0,Wonka Bar - Milk Chocolate,NaN,NaN
1,Wonka Bar - Triple Dazzle Caramel,Lot's O' Nuts,Lot's O' Nuts -> ILLINOIS
2,Wonka Bar - Nutty Crunch Surprise,Sugar Shack,Sugar Shack -> ILLINOIS
3,Wonka Bar -Scrumdiddlyumptious,Secret Factory,Secret Factory -> ILLINOIS
4,Wonka Bar - Triple Dazzle Caramel,Lot's O' Nuts,Lot's O' Nuts -> PENNSYLVANIA


In [15]:
# Filter out crazy outliers (e.g., keep only shipments that took less than 60 days)
df = df[df['Shipping Lead Time'] < 60]

# Now, recalculate your KPIs with the clean, mapped data
route_kpis = df.groupby('Route_to_State').agg(
    Average_Lead_Time=('Shipping Lead Time', 'mean'),
    Route_Volume=('Order ID', 'count')
).reset_index()

route_kpis.sort_values(by='Average_Lead_Time').head(10)

,Route_to_State,Average_Lead_Time,Route_Volume


In [16]:
# Make sure EVERY product name from your dataset is in this list
product_to_factory_map = {
    'Wonka Bar - Triple Dazzle Caramel': "Lot's O' Nuts",
    'Wonka Bar - Nutty Crunch Surprise': "Sugar Shack",
    'Wonka Bar -Scrumdiddlyumptious': "Secret Factory",
    'Wonka Bar - Milk Chocolate': "The Other Factory", # <-- ADD THIS (Use the correct factory name)
    # Add any other missing products here...
}

# Re-run the mapping
df['Factory'] = df['Product Name'].map(product_to_factory_map)
df['Route_to_State'] = df['Factory'] + " -> " + df['State/Province']

In [17]:
# Look at the minimum, maximum, and average lead times in your dataset
df['Shipping Lead Time'].describe()

count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: Shipping Lead Time, dtype: float64

In [18]:
import pandas as pd

# 1. Reload the data from scratch to get our rows back!
df = pd.read_csv('Nassau Candy Distributor.csv')

# 2. Safely process the dates
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True, format='mixed')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], dayfirst=True, format='mixed')

# 3. Calculate the lead time
df['Shipping Lead Time'] = (df['Ship Date'] - df['Order Date']).dt.days

# 4. Standardize states so the route names look clean
df['State/Province'] = df['State/Province'].str.upper()

# 5. Apply your updated mapping
product_to_factory_map = {
    'Wonka Bar - Triple Dazzle Caramel': "Lot's O' Nuts",
    'Wonka Bar - Nutty Crunch Surprise': "Sugar Shack",
    'Wonka Bar -Scrumdiddlyumptious': "Secret Factory",
    'Wonka Bar - Milk Chocolate': "The Other Factory"
    # (Add any others if you found more unique names!)
}
df['Factory'] = df['Product Name'].map(product_to_factory_map)

# 6. Create the Route column
df['Route_to_State'] = df['Factory'] + " -> " + df['State/Province']

# 7. Check the true spread of the lead times!
df['Shipping Lead Time'].describe()

count    10194.000000
mean      1320.841868
std        262.444892
min        904.000000
25%       1271.000000
50%       1274.000000
75%       1638.000000
max       1642.000000
Name: Shipping Lead Time, dtype: float64

In [19]:
# Let's see exactly what Pandas thinks the years are
print("Order Years:", df['Order Date'].dt.year.unique())
print("Ship Years:", df['Ship Date'].dt.year.unique())

# Let's look at the first 10 rows to see the gap visually
df[['Order Date', 'Ship Date', 'Shipping Lead Time']].head(10)

Order Years: [2024 2025]
Ship Years: [2026 2027 2028 2029 2030]


,Order Date,Ship Date,Shipping Lead Time
0,2024-01-03,2026-06-30,909
1,2024-01-04,2026-07-01,909
2,2024-01-04,2026-07-01,909
3,2024-01-04,2026-07-01,909
4,2024-01-05,2026-07-05,912
5,2024-01-06,2026-07-03,909
6,2024-01-06,2026-07-03,909
7,2024-01-06,2026-06-30,906
8,2024-01-06,2026-07-03,909
9,2024-01-06,2026-07-03,909


In [20]:
# We are skipping the filter and analyzing the dataset as-is

# 1. Calculate Route KPIs (Aggregating by Route to State)
route_kpis = df.groupby('Route_to_State').agg(
    Average_Lead_Time=('Shipping Lead Time', 'mean'),
    Route_Volume=('Order ID', 'count')
).reset_index()

# 2. Sort to see the 10 fastest routes (Efficiency Benchmarking)
fastest_routes = route_kpis.sort_values(by='Average_Lead_Time').reset_index(drop=True)

# 3. Display the top 10 most efficient routes
fastest_routes.head(10)

,Route_to_State,Average_Lead_Time,Route_Volume
0,Lot's O' Nuts -> MAINE,908.000000,1
1,The Other Factory -> MAINE,908.000000,1
2,The Other Factory -> MONTANA,911.000000,1
3,Secret Factory -> NEW HAMPSHIRE,1089.500000,4
4,Lot's O' Nuts -> NEW HAMPSHIRE,1089.500000,4
5,Secret Factory -> LOUISIANA,1091.000000,4
6,The Other Factory -> MISSISSIPPI,1121.500000,12
7,Sugar Shack -> UTAH,1152.000000,6
8,Lot's O' Nuts -> SOUTH DAKOTA,1152.000000,3
9,Lot's O' Nuts -> KANSAS,1152.166667,6


In [21]:
# Sort descending to find the slowest routes
slowest_routes = route_kpis.sort_values(by='Average_Lead_Time', ascending=False).reset_index(drop=True)

# Display the bottom 10
st.write("10 Least Efficient Routes:")
slowest_routes.head(10)

NameError: name 'st' is not defined

In [22]:
# Sort descending to find the slowest routes
slowest_routes = route_kpis.sort_values(by='Average_Lead_Time', ascending=False).reset_index(drop=True)

# Display the bottom 10
print("10 Least Efficient Routes:")
slowest_routes.head(10)

10 Least Efficient Routes:


,Route_to_State,Average_Lead_Time,Route_Volume
0,Secret Factory -> SOUTH DAKOTA,1639.750000,4
1,Secret Factory -> NORTH DAKOTA,1639.000000,3
2,The Other Factory -> WEST VIRGINIA,1639.000000,1
3,Sugar Shack -> WEST VIRGINIA,1639.000000,1
4,Lot's O' Nuts -> WEST VIRGINIA,1639.000000,1
5,Sugar Shack -> VERMONT,1638.666667,3
6,The Other Factory -> KANSAS,1637.666667,3
7,Lot's O' Nuts -> NORTH DAKOTA,1637.000000,2
8,Sugar Shack -> NORTH DAKOTA,1637.000000,1
9,Secret Factory -> WEST VIRGINIA,1635.000000,1


In [23]:
# Group by Ship Mode to see the average lead time and total orders for each method
ship_mode_kpis = df.groupby('Ship Mode').agg(
    Average_Lead_Time=('Shipping Lead Time', 'mean'),
    Total_Shipments=('Order ID', 'count')
).reset_index()

ship_mode_kpis.sort_values(by='Average_Lead_Time')

,Ship Mode,Average_Lead_Time,Total_Shipments
3,Standard Class,1314.334641,6120
2,Second Class,1323.845376,1979
1,Same Day,1333.442413,547
0,First Class,1338.275840,1548


In [24]:
# Calculate the average volume across all routes to establish a baseline
avg_volume = route_kpis['Route_Volume'].mean()

# Filter for routes that have higher-than-average volume, then sort by the slowest lead times
bottlenecks = route_kpis[route_kpis['Route_Volume'] > avg_volume].sort_values(by='Average_Lead_Time', ascending=False)

# Display the top 10 worst bottlenecks
print("Top 10 Geographic Bottlenecks:")
bottlenecks.head(10)

Top 10 Geographic Bottlenecks:


,Route_to_State,Average_Lead_Time,Route_Volume
51,Lot's O' Nuts -> WASHINGTON,1389.690265,113
100,Secret Factory -> TENNESSEE,1389.078947,38
34,Lot's O' Nuts -> NORTH CAROLINA,1379.454545,55
147,Sugar Shack -> PENNSYLVANIA,1370.643564,101
158,Sugar Shack -> WASHINGTON,1363.588235,85
212,The Other Factory -> WASHINGTON,1361.111111,108
41,Lot's O' Nuts -> PENNSYLVANIA,1354.098361,122
105,Secret Factory -> WASHINGTON,1352.910891,101
167,The Other Factory -> COLORADO,1349.710526,38
141,Sugar Shack -> NORTH CAROLINA,1347.836735,49
